# Sistemas Inteligentes I
## Búsqueda adversarial: poda Alfa–Beta

**Autor:** Jairo I. Vélez B.

---


# 1. Punto de partida: el problema de Minimax

Minimax supone que:

- MAX intenta maximizar;
- MIN intenta minimizar;
- ambos jugadores actúan racionalmente.

El problema es que, para garantizar su decisión, Minimax puede explorar aproximadamente:

$$O(b^m)$$

nodos.

Sin embargo, algunas ramas pueden resultar irrelevantes.

La idea central de Alfa–Beta es:

> **dejar de explorar una rama cuando ya sabemos que no puede mejorar la decisión de un jugador.**

La poda no cambia el resultado de Minimax.  
Solo intenta obtenerlo explorando menos nodos.

# 2. Recordatorio: un árbol de juego

Utilizaremos inicialmente el mismo tipo de representación:

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Minimax calcula:

- `B = 2`
- `C = 1`
- `D = 6`

y finalmente:

$$A=\max(2,1,6)=6$$

In [ ]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades

# 3. ¿Qué representan alfa y beta?

Durante la búsqueda mantenemos dos límites.

### Alfa — $\alpha$

Es el mejor valor que **MAX puede garantizar hasta el momento**.

Inicialmente:

$$\alpha=-\infty$$

### Beta — $\beta$

Es el mejor valor que **MIN puede garantizar hasta el momento**.

Inicialmente:

$$\beta=+\infty$$

Durante la búsqueda:

- MAX actualiza $\alpha$;
- MIN actualiza $\beta$.

Cuando ocurre:

$$\boxed{\alpha \geq \beta}$$

podemos realizar una **poda**.

# 4. Intuición de una poda

Suponga que MAX ya dispone de una alternativa con valor `6`.

Ahora explora otra rama cuyo turno pertenece a MIN.

Si MIN encuentra dentro de esa rama una opción con valor `4`, sabemos que podrá forzar:

$$valor \leq 4$$

MAX ya dispone de `6`, por lo que nunca elegirá una alternativa que termine en
`4` o menos.

Por tanto:

> **el resto de esa rama ya no puede cambiar la decisión de MAX.**

Podemos dejar de explorarla.

# 5. Implementación de Alfa–Beta

La estructura es muy similar a Minimax.

La diferencia está en que conservamos y actualizamos los límites
$\alpha$ y $\beta$.

In [ ]:
from math import inf

def alfa_beta(nodo, es_max, arbol, utilidades, alfa=-inf, beta=inf):
    if nodo in utilidades:
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta(hijo, False, arbol, utilidades, alfa, beta)
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta(hijo, True, arbol, utilidades, alfa, beta)
            )

            beta = min(beta, valor)

            if alfa >= beta:
                break

        return valor


alfa_beta("A", True, arbol, utilidades)

# 6. Comparar Alfa–Beta con Minimax

Primero implementaremos una versión sencilla de Minimax.

In [ ]:
def minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo]

    valores = [
        minimax(hijo, not es_max, arbol, utilidades)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


print("Minimax   :", minimax("A", True, arbol, utilidades))
print("Alfa-Beta :", alfa_beta("A", True, arbol, utilidades))

# 7. Alfa–Beta paso a paso

Ahora imprimiremos:

- nodo visitado;
- jugador;
- valor de $\alpha$;
- valor de $\beta$;
- momento en el que se produce una poda.

In [ ]:
def alfa_beta_debug(
    nodo,
    es_max,
    arbol,
    utilidades,
    alfa=-inf,
    beta=inf,
    profundidad=0
):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(
            f"{sangria}{nodo}: terminal = {utilidades[nodo]} "
            f"[α={alfa}, β={beta}]"
        )
        return utilidades[nodo]

    print(
        f"{sangria}{nodo}: {jugador} "
        f"[α={alfa}, β={beta}]"
    )

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                False,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = max(valor, valor_hijo)
            alfa = max(alfa, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                True,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = min(valor, valor_hijo)
            beta = min(beta, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor


alfa_beta_debug("A", True, arbol, utilidades)

# 8. Un ejemplo diseñado para observar podas

El orden de los valores del árbol anterior no siempre produce una poda muy visible.

Usaremos ahora este árbol:

```text
                         A (MAX)
                    /             \
               B (MIN)           C (MIN)
              /      \           /      \
          D(MAX)   E(MAX)    F(MAX)    G(MAX)
           3  5     6  9      1  2      0 -1
```

La exploración se realiza de izquierda a derecha.

In [ ]:
arbol_poda = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_poda = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

print("Minimax:", minimax("A", True, arbol_poda, utilidades_poda))
print()
alfa_beta_debug("A", True, arbol_poda, utilidades_poda)

# 9. Medir el ahorro de exploración

Para comparar los algoritmos contabilizaremos:

- nodos visitados;
- hojas evaluadas;
- podas realizadas.

In [ ]:
def minimax_contando(nodo, es_max, arbol, utilidades, estadisticas):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(
            hijo,
            not es_max,
            arbol,
            utilidades,
            estadisticas
        )
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


def alfa_beta_contando(
    nodo,
    es_max,
    arbol,
    utilidades,
    estadisticas,
    alfa=-inf,
    beta=inf
):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta_contando(
                    hijo,
                    False,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta_contando(
                    hijo,
                    True,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor


stats_minimax = {"visitados": 0, "hojas": 0}
stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

valor_mm = minimax_contando(
    "A", True, arbol_poda, utilidades_poda, stats_minimax
)

valor_ab = alfa_beta_contando(
    "A", True, arbol_poda, utilidades_poda, stats_ab
)

print("Valor Minimax:", valor_mm)
print("Valor Alfa-Beta:", valor_ab)
print()
print("Minimax:", stats_minimax)
print("Alfa-Beta:", stats_ab)

### Preguntas de análisis

1. ¿Ambos algoritmos producen el mismo valor?
2. ¿Cuántas hojas evita evaluar Alfa–Beta?
3. ¿Qué información permite justificar una poda?
4. ¿Podar significa que la rama sea necesariamente mala?
5. ¿Podría una rama podada contener valores muy altos o muy bajos?

### Respuestas — Preguntas de análisis (sección 9)

1. **¿Ambos algoritmos producen el mismo valor?** Sí. Alfa-Beta y Minimax devuelven `5` en este árbol. La poda no cambia el resultado, solo evita explorar ramas que no aportan a la decisión.

2. **¿Cuántas hojas evita evaluar Alfa-Beta?** Minimax evalúa las 8 hojas. Alfa-Beta evalúa 5 hojas (D1, D2, E1, F1, F2) y se salta 3 (E2, G1, G2). O sea, se ahorra 3 evaluaciones de hoja.

3. **¿Qué información permite justificar una poda?** El hecho de que ya conozco una jugada mejor para el jugador de arriba. Cuando alfa >= beta, MAX ya tiene garantizado por otra rama algo >= alfa, y MIN nunca le va a dejar tomar algo peor que beta. Como esta rama tiene valor >= alfa, MIN no me la va a dejar de todas formas, así que no vale la pena seguir mirándola.

4. **¿Podar significa que la rama sea necesariamente mala?** No. Puede que la rama tenga valores excelentes, pero como el jugador de arriba ya tiene algo mejor por otro lado, no importa terminar de explorarla.

5. **¿Podría una rama podada contener valores muy altos o muy bajos?** Sí. Como no la exploramos completa, no sabemos qué había ahí. Lo importante es que el resultado global es el mismo que en Minimax: los valores podados no habrían cambiado la decisión.

# 10. El orden de exploración importa

Alfa–Beta es especialmente eficaz cuando primero examinamos las jugadas más prometedoras.

En el mejor caso, su complejidad puede aproximarse a:

$$O(b^{m/2})$$

en lugar de:

$$O(b^m)$$

Esto significa que, con un buen ordenamiento, puede ser posible explorar
aproximadamente el doble de profundidad usando recursos comparables.

Sin embargo:

> **Alfa–Beta sigue siendo correcto independientemente del orden.  
> El orden afecta cuánto poda, no el valor final.**

## 10.1 Comparar dos órdenes del mismo árbol

Crearemos dos versiones:

- una con un orden favorable;
- otra con un orden menos favorable.

Los valores terminales son exactamente los mismos.

In [ ]:
arbol_buen_orden = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D2", "D1"],
    "E": ["E2", "E1"],
    "F": ["F2", "F1"],
    "G": ["G1", "G2"],
}

arbol_mal_orden = {
    "A": ["C", "B"],
    "B": ["E", "D"],
    "C": ["G", "F"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G2", "G1"],
}

def medir_alfa_beta(arbol):
    stats = {"visitados": 0, "hojas": 0, "podas": 0}
    valor = alfa_beta_contando(
        "A",
        True,
        arbol,
        utilidades_poda,
        stats
    )
    return valor, stats


print("Orden 1:", medir_alfa_beta(arbol_buen_orden))
print("Orden 2:", medir_alfa_beta(arbol_mal_orden))

### Preguntas de análisis

1. ¿Cambió el valor final?
2. ¿Cambió el número de nodos visitados?
3. ¿Por qué conocer primero una buena jugada ayuda a podar?
4. ¿Cómo podría un programa real ordenar las jugadas antes de examinarlas?

### Respuestas — Preguntas de análisis (sección 10.1)

1. **¿Cambió el valor final?** No. En los dos órdenes A vale 5. Alfa-Beta siempre da el mismo resultado que Minimax, sin importar el orden en que se examinen las jugadas.

2. **¿Cambió el número de nodos visitados?** Sí. Con el buen orden se visitan menos nodos que con el mal orden. Cuando primero se ven jugadas malas, alfa y beta se ajustan más lento y se poda menos.

3. **¿Por qué conocer primero una buena jugada ayuda a podar?** Porque si desde el principio encuentro una jugada buena para MAX, alfa sube rápido. Con alfa alto, es más fácil que se cumpla `alfa >= beta` en las ramas siguientes, así que se podan antes. Lo mismo al revés para MIN con beta.

4. **¿Cómo podría un programa real ordenar las jugadas antes de examinarlas?** Con heurísticas del dominio. En ajedrez, por ejemplo, se examinan primero las capturas, los jaques y las jugadas que fueron buenas en posiciones parecidas (killer moves, historial de búsqueda). En juegos más simples se puede usar una evaluación rápida del estado para ordenar.

# 11. Caso aplicado: juego de las piedras

Retomaremos el juego:

- hay una pila de piedras;
- cada jugador puede retirar `1`, `2` o `3`;
- quien retira la última piedra gana.

Compararemos Minimax y Alfa–Beta sobre el mismo juego.

In [ ]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


def minimax_piedras_contando(piedras, turno_max, stats):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    valores = [
        minimax_piedras_contando(
            piedras - retirar,
            not turno_max,
            stats
        )
        for retirar in movimientos_validos(piedras)
    ]

    return max(valores) if turno_max else min(valores)


def alfa_beta_piedras(
    piedras,
    turno_max,
    stats,
    alfa=-inf,
    beta=inf
):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    if turno_max:
        valor = -inf

        for retirar in movimientos_validos(piedras):
            valor = max(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    False,
                    stats,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for retirar in movimientos_validos(piedras):
            valor = min(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    True,
                    stats,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor

## 11.1 Comparación experimental

In [ ]:
for piedras in [6, 8, 10, 12]:
    stats_mm = {"visitados": 0, "hojas": 0}
    stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

    valor_mm = minimax_piedras_contando(
        piedras, True, stats_mm
    )

    valor_ab = alfa_beta_piedras(
        piedras, True, stats_ab
    )

    print(f"\n{piedras} piedras")
    print("  Minimax   :", valor_mm, stats_mm)
    print("  Alfa-Beta :", valor_ab, stats_ab)

# 12. Obtener la mejor jugada con Alfa–Beta

En una aplicación real necesitamos devolver una **acción**, no solo el valor.

In [ ]:
def mejor_jugada_alfa_beta_piedras(piedras):
    mejor_valor = -inf
    mejor_movimiento = None
    alfa = -inf
    beta = inf

    for retirar in movimientos_validos(piedras):
        stats = {"visitados": 0, "hojas": 0, "podas": 0}

        valor = alfa_beta_piedras(
            piedras - retirar,
            False,
            stats,
            alfa,
            beta
        )

        if valor > mejor_valor:
            mejor_valor = valor
            mejor_movimiento = retirar

        alfa = max(alfa, mejor_valor)

    return mejor_movimiento, mejor_valor


for piedras in range(1, 11):
    movimiento, valor = mejor_jugada_alfa_beta_piedras(piedras)

    print(
        f"{piedras:2d} piedras -> "
        f"retirar {movimiento}, valor {valor}"
    )

In [ ]:
LINEAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),   # filas
    (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columnas
    (0, 4, 8), (2, 4, 6),              # diagonales
]

def acciones(tablero):
    return [i for i in range(9) if tablero[i] == " "]

def resultado(tablero, accion, jugador):
    nuevo = list(tablero)
    nuevo[accion] = jugador
    return tuple(nuevo)

def ganador(tablero):
    for a, b, c in LINEAS:
        if tablero[a] != " " and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    return None

def terminal(tablero):
    return ganador(tablero) is not None or " " not in tablero

def utilidad(tablero):
    g = ganador(tablero)
    if g == "X":
        return 1
    if g == "O":
        return -1
    return 0

def alfa_beta_tictactoe(tablero, es_max, alfa=-inf, beta=inf):
    if terminal(tablero):
        return utilidad(tablero)
    jugador = "X" if es_max else "O"
    if es_max:
        valor = -inf
        for accion in acciones(tablero):
            siguiente = resultado(tablero, accion, jugador)
            valor = max(valor, alfa_beta_tictactoe(siguiente, False, alfa, beta))
            alfa = max(alfa, valor)
            if alfa >= beta:
                break
        return valor
    else:
        valor = inf
        for accion in acciones(tablero):
            siguiente = resultado(tablero, accion, jugador)
            valor = min(valor, alfa_beta_tictactoe(siguiente, True, alfa, beta))
            beta = min(beta, valor)
            if alfa >= beta:
                break
        return valor

In [ ]:
# Prueba de la implementación con un tablero donde X puede ganar

tablero_prueba = (
    "X", "X", " ",
    "O", "O", " ",
    " ", " ", " ",
)

for i in range(0, 9, 3):
    fila = ""
    for x in tablero_prueba[i:i+3]:
        if x == " ":
            fila += ". "
        else:
            fila += x + " "
    print(fila)

print()
print("Valor alfa-beta (turno X):", alfa_beta_tictactoe(tablero_prueba, True))

# Tablero vacio: con juego perfecto, tres en raya siempre termina en empate
print()
tablero_vacio = tuple(" " for _ in range(9))
print("Valor alfa-beta desde tablero vacio (turno X):", alfa_beta_tictactoe(tablero_vacio, True))

### Uso de IA generativa

- **Herramienta utilizada:** Claude (Anthropic).
- **Propósito de uso:** apoyo en la redacción y revisión de estilo de las respuestas a las preguntas de análisis.
- **Partes en las que fue empleada:** respuestas de las secciones 9 y 10.1.